# 09 — Dashboard Design & Data Population (Phase 9)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 9 — Streamlit Dashboard

## 1. Project Overview

The dashboard (`dashboard/app.py`) needs to be a **read-only, fast** window
onto already-processed DGCA publications. Those two requirements collide
with one fact: a local LLM analysis call (Phase 8A) takes 60-90 seconds.
Running that live on every dashboard page view would make the UI
unusable, so this notebook does the one-time (well, re-runnable) work of
**persisting** AI analysis and comparison results into the same PostgreSQL
database the dashboard reads from -- so the dashboard itself does no AI
work at all, just fast SQL reads.

```
Existing PostgreSQL database (PostgreSQL via `DATABASE_URL`)
        |
        v
pipeline.storage.ensure_schema_migrations()   <- additive columns only,
        |                                         Phase 3-8's schema/rows untouched
        v
pipeline.documents / pipeline.comparison / pipeline.analysis   <- UNCHANGED,
        |                                                          same functions
        |                                                          Phase 7/8A already tested
        v
pipeline.storage.save_analysis_result()   <- new, writes results back
        |
        v
pipeline.storage.get_readonly_connection() / get_publications() / get_summary_stats()
        |
        v
dashboard/app.py (Streamlit, read-only)
```

**Nothing about the AI analysis architecture changes here.** This notebook
calls `pipeline.analysis.generate_ai_analysis()` exactly as Phase 8A left
it -- `LLM_PROVIDER=local` (Ollama), automatic fallback on failure, no
OpenAI calls. The only new code is the *persistence* layer
(`pipeline/storage.py` additions) and this notebook's population loop.

## Dashboard section -> data source design

| Dashboard section | Backed by |
|---|---|
| Overview | `storage.get_summary_stats()` |
| Latest Publications (table, search/filter) | `storage.get_publications(...)` |
| Publication Detail | `storage.get_publication_by_id()` |
| Document Actions | `publication_url` / `pdf_url` columns already in the DB (never invented) |
| Old vs New Comparison | `comparison_summary_json` + `previous_publication_id`, populated below |
| Search and Filters | SQL `WHERE` clauses inside `get_publications()` -- not reimplemented in the dashboard |
| AI Analysis transparency | `analysis_mode` column, always shown next to the analysis in the UI |

Real DGCA publications and the Phase 7 synthetic test fixture
(`e2e-test-car-x-rev1`/`rev2`/`independent-doc`) are populated and clearly
distinguished (`publication_id` prefix), never merged into one undistinguished
list -- the dashboard filters test fixtures out of the main views entirely
and shows them only in a separate, clearly labeled section.


## 2. Imports

In [ ]:
import sys
import time
from pathlib import Path

_project_root_for_import = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_project_root_for_import) not in sys.path:
    sys.path.insert(0, str(_project_root_for_import))

from pipeline.config import load_config
from pipeline.utils import configure_logging
from pipeline import storage
from pipeline import comparison
from pipeline import analysis

print("Imports OK -- pipeline package loaded.")


## 3. Configuration

Same configuration as every prior phase. `LLM_PROVIDER` stays whatever
Phase 8A left it as (`local`, the default) -- this notebook does not
override it. No OpenAI calls, no Azure requirement.


In [ ]:
PROJECT_ROOT = _project_root_for_import
ENV_PATH = PROJECT_ROOT / "config" / ".env"
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
LOG_DIR = PROJECT_ROOT / "logs"

config = load_config(ENV_PATH)
logger = configure_logging(LOG_DIR, "dashboard_population")

print(f"LLM_PROVIDER: {config.llm_provider}")
print(f"OLLAMA_MODEL: {config.ollama_model}")
print(f"EMAIL_PROVIDER: {config.email_provider} (unused in this notebook -- no emails are sent)")


## 4. Schema Migration Check

Additive only -- adds AI-analysis columns if they're not already present
(they will be, if Phase 9's `pipeline/storage.py` changes have already run
once). Existing rows and every pre-Phase-9 column are untouched.


In [ ]:
storage.init_db()

conn = storage.get_connection()  # write connection -- this notebook populates data
cols = [r["name"] for r in conn.execute("SELECT column_name AS name FROM information_schema.columns WHERE table_schema = current_schema() AND table_name = 'publications' ORDER BY ordinal_position")]
print(f"Columns on publications table ({len(cols)}):")
for c in cols:
    print(f"  - {c}")


## 5. Populate Real AI Analysis

For every **real** DGCA publication already in the database (from Phases
2-4), this: looks up its extracted text (if any), looks for a previous
version and computes a comparison (if one exists), runs
`generate_ai_analysis()` (local LLM, with its existing automatic fallback),
and persists the result. This reuses Phase 7's per-publication logic
pattern exactly -- **no email step**, since the dashboard is read-only and
this notebook's job is only to populate data for it to display.

This takes roughly a minute per publication with extracted text (local LLM
inference) and is near-instant for the ones without extracted text
(`needs_ocr` -- `generate_ai_analysis()` already handles that gracefully as
a "degraded" result, same as Phase 8A).


In [ ]:
def populate_analysis_for(pub_id: str) -> "analysis.AIAnalysisResult":
    row = conn.execute("SELECT * FROM publications WHERE publication_id = %s", (pub_id,)).fetchone()
    text_path = DOCUMENTS_DIR / f"{pub_id}.txt"
    extracted_text = text_path.read_text(encoding="utf-8") if text_path.exists() else None

    previous_row = comparison.find_previous_version(conn, pub_id)
    previous_id = None
    comparison_result = None
    if previous_row is not None:
        previous_id = previous_row["publication_id"]
        old_text_path = DOCUMENTS_DIR / f"{previous_id}.txt"
        if old_text_path.exists() and extracted_text:
            _diff, comparison_result = comparison.compare_documents(
                old_text_path.read_text(encoding="utf-8"), extracted_text, config
            )

    result = analysis.generate_ai_analysis(row, extracted_text, comparison_result, config)
    storage.save_analysis_result(conn, pub_id, result, previous_publication_id=previous_id, comparison=comparison_result)
    return result


real_rows = [
    r for r in storage.get_all(conn)
    if not r["publication_id"].startswith("e2e-test") and not r["publication_id"].startswith("test-")
]
print(f"Real DGCA publications to analyze: {len(real_rows)}")


In [ ]:
for row in real_rows:
    pub_id = row["publication_id"]
    t0 = time.time()
    result = populate_analysis_for(pub_id)
    elapsed = time.time() - t0
    print(f"[{pub_id}] mode={result.analysis_mode:<10} impact={result.impact_level or 'n/a':<8} ({elapsed:.1f}s)")

print()
print(f"Populated analysis for {len(real_rows)} real publications.")


## 6. Populate Synthetic Test Fixture (clearly labeled, kept separate)

Real DGCA data has no genuine revision pair (established repeatedly since
Phase 5). Phase 7's synthetic `e2e-test-car-x-rev1`/`rev2` fixture is the
project's only real *previous-version-with-a-real-diff* example, so it's
populated here too -- but its `publication_id` prefix (`e2e-test-`/`test-`)
is what the dashboard uses to keep it entirely out of the main "real data"
views and confined to one clearly labeled demo section, per this phase's
explicit instruction not to mix synthetic and real data.


In [ ]:
test_rows = [r for r in storage.get_all(conn) if r["publication_id"].startswith("e2e-test") or r["publication_id"].startswith("test-")]
print(f"Synthetic test-fixture publications to analyze: {len(test_rows)}")

for row in test_rows:
    pub_id = row["publication_id"]
    t0 = time.time()
    result = populate_analysis_for(pub_id)
    elapsed = time.time() - t0
    print(f"[SYNTHETIC] [{pub_id}] mode={result.analysis_mode:<10} impact={result.impact_level or 'n/a':<8} ({elapsed:.1f}s)")

print()
print(f"Populated analysis for {len(test_rows)} synthetic test-fixture publications.")


## 7. Verify Persisted Data

In [ ]:
conn.commit()
conn.close()

# Re-open read-only, exactly as the dashboard will.
ro_conn = storage.get_readonly_connection()

stats = storage.get_summary_stats(ro_conn)
print("Summary stats (real publications only):")
for k, v in stats.items():
    print(f"  {k}: {v}")

print()
real_pubs = storage.get_publications(ro_conn)
with_analysis = [p for p in real_pubs if p["analysis_mode"] is not None]
print(f"Real publications with a persisted analysis_mode: {len(with_analysis)}/{len(real_pubs)}")

compared = [p for p in real_pubs if p["comparison_summary"] is not None]
print(f"Real publications with a persisted comparison: {len(compared)}")

test_pubs = storage.get_publications(ro_conn, include_test_fixtures=True)
test_only = [p for p in test_pubs if p not in real_pubs]
test_only_analyzed = sum(1 for p in test_only if p["analysis_mode"] is not None)
print(f"Test-fixture publications with a persisted analysis_mode: {test_only_analyzed}/{len(test_only)}")


## 8. Read-Only Connection Safety Check

Structural proof (not just a promise) that the connection the dashboard
will use cannot modify data -- PostgreSQL itself rejects the write at the
driver level.


In [ ]:
from psycopg.errors import ReadOnlySqlTransaction

try:
    ro_conn.execute("UPDATE publications SET status = 'hacked' WHERE 1=1")
    raise AssertionError("Read-only connection allowed a write -- this must not happen.")
except ReadOnlySqlTransaction as exc:
    print(f"Read-only connection correctly rejected a write attempt: {exc.__class__.__name__}: {exc}")

ro_conn.close()
print()
print("Confirmed: dashboard/app.py uses exactly this connection type -- it cannot alter production data.")


## 9. Results

In [ ]:
print("=" * 70)
print("PHASE 9 DATA POPULATION RESULTS")
print("=" * 70)
print(f"Real DGCA publications analyzed  : {len(real_rows)}")
print(f"Synthetic test fixtures analyzed : {len(test_rows)}")
print(f"AI provider used                 : {config.llm_provider}")
print(f"Database                         : PostgreSQL (DATABASE_URL)")
print("=" * 70)


## 10. Next Steps

Data is now persisted and ready for `dashboard/app.py` to read via
`pipeline.storage.get_readonly_connection()`. That app contains no
database/business logic of its own -- every query it needs already exists
in `pipeline/storage.py` (`get_summary_stats`, `get_publications`,
`get_publication_by_id`).

Not started yet (by design, per this phase's instructions): a scheduler /
automatic monitoring. The dashboard is a read-only view over whatever this
notebook (or a future automated run of the same population logic) has
already persisted.
